# KhadSan AI รอบ 1 : Model Tuning (7 การทดลอง)

เทรน YOLOv8n ตีกรอบผลไม้ 5 ชนิด (แอปเปิล กล้วย ส้ม ทับทิม มะนาว) × สด/เน่า = 10 คลาส
dataset หลัก : [Fruits Quality](https://universe.roboflow.com/alex-day-qzfwa/fruits-quality-qtdtf) (Roboflow, CC BY 4.0)

**การทดลอง** (เริ่มจาก Baseline แล้วเปลี่ยนทีละอย่าง ใช้ train/val/test ชุดเดียวกันทุกตัว)

| การทดลอง | เปลี่ยนอะไร |
|---|---|
| 1_scratch_sgd | เทรนจากศูนย์ |
| 2_pretrained_sgd | Baseline : Pretrained, SGD, LR 0.01 คงที่, Batch 16, Weight decay 0.0005 |
| 3_adam | Optimizer = Adam (LR 0.001) |
| 4_rmsprop | Optimizer = RMSProp (LR 0.001) |
| 5_cosine_lr | Learning rate แบบ Cosine |
| 6_batch32 | Batch size 32 |
| 7_no_weight_decay | ไม่ใช้ L2 Weight decay |

ทุกการทดลอง : สูงสุด 60 epoch, Early Stopping patience 10, รูป 640

**ก่อนกด Run all**
1. push โฟลเดอร์ `train/` ล่าสุดขึ้น GitHub
2. Runtime > Change runtime type > **T4 GPU**
3. dataset หลัก : ตั้ง Colab Secret ชื่อ `ROBOFLOW_API_KEY` **หรือ** อัปโหลด `FruitsQuality.zip` ไว้ใน Drive โฟลเดอร์ `KhadSan_detect`
4. (แนะนำ) อัปโหลด `ExternalTest.zip` ไว้ใน Drive โฟลเดอร์ `KhadSan_detect`

ใช้เวลาราว 2-3 ชั่วโมง ถ้าหลุดกลางทาง กด Run all ใหม่ การทดลองที่เสร็จแล้วจะถูกข้าม

## 1) เช็กว่าได้ GPU
ถ้าขึ้นว่าไม่เจอ GPU : Runtime > Change runtime type > **T4 GPU** > Save แล้วกด Run all ใหม่

In [ ]:
import torch
if not torch.cuda.is_available():
    raise Exception("ไม่เจอ GPU ไปที่ Runtime > Change runtime type > เลือก T4 GPU แล้วกด Run all ใหม่")
print("ได้ GPU:", torch.cuda.get_device_name(0))

## 2) ติดตั้งไลบรารี
ล็อกรุ่น ultralytics ให้ตรงกับที่ทีมใช้ (8.4.191) ผลจะได้เทียบกันได้

In [ ]:
!pip install -q ultralytics==8.4.191 roboflow onnx onnxruntime
!yolo settings sync=False
import ultralytics
print("ultralytics", ultralytics.__version__)
if ultralytics.__version__ != "8.4.191":
    raise Exception("รุ่น ultralytics ไม่ตรง ให้ Runtime > Restart session แล้วกด Run all ใหม่")

## 3) ดึงโค้ดเทรนจาก GitHub ของทีม
ต้อง **push โค้ดล่าสุดขึ้น GitHub ก่อน** ช่องนี้จะเช็กให้ว่าโค้ดที่ดึงมาเป็นรุ่นที่ใช้กับรอบนี้ได้

In [ ]:
import os
REPO = "https://github.com/srirat-intalai/khadsan-ai"

# ดึงครั้งแรก = clone / รันซ้ำ = อัปเดตเป็นโค้ดล่าสุด
if os.path.isdir("/content/khadsan-ai"):
    !git -C /content/khadsan-ai pull -q
else:
    !git clone -q {REPO} /content/khadsan-ai

CODE = "/content/khadsan-ai/train"
!git -C /content/khadsan-ai log -1 --format="โค้ดรุ่นล่าสุด: %h %s (%ad)" --date=short

# เช็กว่าโค้ดมีของที่รอบนี้ต้องใช้ (ถ้าไม่มี แปลว่ายังไม่ได้ push โค้ดล่าสุด)
train_code = open(CODE + "/train_yolo.py", encoding="utf-8").read()
prep_code = open(CODE + "/prepare_data.py", encoding="utf-8").read()

for need in ["7_no_weight_decay", "--only", "set_yscale"]:
    if need not in train_code:
        raise Exception("โค้ดใน GitHub ยังเป็นรุ่นเก่า (ไม่มี " + need + ") ให้ push โค้ดล่าสุดก่อน")

print("โค้ดครบ พร้อมเทรน")

## 4) ต่อ Google Drive แล้วโหลด dataset
ผลเก็บใน Drive โฟลเดอร์ `KhadSan_detect` ถ้าในโฟลเดอร์มีผลของรอบเก่าอยู่ จะถูกข้าม (ไม่เทรนซ้ำ)

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

OUT = "/content/drive/MyDrive/KhadSan_detect"
os.makedirs(OUT, exist_ok=True)
print("ผลจะเก็บที่", OUT)

# ----- dataset หลัก (Fruits Quality) : ใช้ zip ใน Drive ถ้ามี ไม่มีก็โหลดจาก Roboflow ด้วย API key -----
SRC = "/content/FruitsQuality"
if os.path.exists(OUT + "/FruitsQuality.zip"):
    print("ใช้ FruitsQuality.zip จาก Drive")
    !unzip -q -o "{OUT}/FruitsQuality.zip" -d /content/zip_tmp
    for root, dirs, files in os.walk("/content/zip_tmp"):
        if "data.yaml" in files:
            SRC = root
            break
else:
    from google.colab import userdata
    from roboflow import Roboflow
    rf = Roboflow(api_key=userdata.get("ROBOFLOW_API_KEY"))
    project = rf.workspace("alex-day-qzfwa").project("fruits-quality-qtdtf")
    project.version(1).download("yolov8", location=SRC)

# เช็ก dataset หลัก : ต้องมี data.yaml และรูปรวมกันประมาณ 2,583 รูป
if not os.path.exists(SRC + "/data.yaml"):
    raise Exception("ไม่เจอ data.yaml ใน dataset หลัก เช็กไฟล์ zip หรือ API key")
count = 0
for part in ["train", "valid", "test"]:
    if os.path.isdir(SRC + "/" + part + "/images"):
        count = count + len(os.listdir(SRC + "/" + part + "/images"))
print("dataset หลัก", count, "รูป (ควรได้ 2583)")
if count != 2583:
    print("!!! จำนวนรูปไม่ตรงกับที่ทีมใช้ ผลอาจต่างจากเดิม")

# ----- ชุดทดสอบภายนอก (ExternalTest.zip) : 6 คลาส คลาสละ 100 รูป -----
EXTERNAL = ""
if os.path.exists(OUT + "/ExternalTest.zip"):
    !unzip -q -o "{OUT}/ExternalTest.zip" -d /content/external_tmp
    for root, dirs, files in os.walk("/content/external_tmp"):
        if "Apple_Good" in dirs:
            EXTERNAL = root
            break
    total = 0
    for c in sorted(os.listdir(EXTERNAL)):
        if os.path.isdir(EXTERNAL + "/" + c):
            n = len(os.listdir(EXTERNAL + "/" + c))
            total = total + n
            print("  ", c, n, "รูป")
    print("ชุดทดสอบภายนอก", total, "รูป (ควรได้ 600)")
else:
    print("ไม่มี ExternalTest.zip ข้ามการทดสอบภายนอก (แนะนำให้อัปโหลด)")

# เตือนถ้ามีผลเก่าค้างอยู่ (การทดลองที่มีผลแล้วจะถูกข้าม)
if os.path.exists(OUT + "/results/summary.csv"):
    print("!!! มีผลเก่าใน", OUT, "การทดลองที่เสร็จแล้วจะถูกข้าม ถ้าอยากเทรนใหม่ทั้งหมด ให้เปลี่ยนชื่อโฟลเดอร์เก่าก่อน")

## 5) เตรียมข้อมูล
เก็บแค่ 5 ชนิด, จับกลุ่มรูปที่คล้ายกันไว้ชุดเดียวกัน (กัน data leakage), แบ่งใหม่ 70 / 15 / 15

In [ ]:
!python "{CODE}/prepare_data.py" --src "{SRC}" --out /content/data_5fruits

# เช็กผล : ต้องมี data.yaml และรูปครบ 3 ชุด
if not os.path.exists("/content/data_5fruits/data.yaml"):
    raise Exception("เตรียมข้อมูลไม่สำเร็จ ดูข้อความ error ด้านบน")
for part in ["train", "val", "test"]:
    print(part, len(os.listdir("/content/data_5fruits/" + part + "/images")), "รูป")
!cp /content/data_5fruits/dataset_counts.csv "{OUT}/"

## 6) เริ่มเทรน 7 การทดลอง

In [ ]:
!python "{CODE}/train_yolo.py" --data /content/data_5fruits/data.yaml --only 1_scratch_sgd,2_pretrained_sgd,3_adam,4_rmsprop,5_cosine_lr,6_batch32,7_no_weight_decay --out "{OUT}" --external "{EXTERNAL}"

## 7) ดูผล

In [ ]:
import glob
from IPython.display import Image, Markdown, display

display(Markdown(open(OUT + "/results/summary.md", encoding="utf-8").read()))
print(open(OUT + "/models/class_info.json", encoding="utf-8").read())

for name in ["compare_scratch_vs_pretrained", "compare_optimizers", "compare_lr_fixed_vs_cosine", "compare_batch16_vs_32", "compare_weight_decay", "compare_all", "confusion_matrix_normalized_best"]:
    file = OUT + "/results/" + name + ".png"
    if os.path.exists(file):
        print(name)
        display(Image(file))

print(open(OUT + "/results/per_class_best.csv", encoding="utf-8").read())

# ผลทดสอบกับรูปจากแหล่งอื่น ของทุกการทดลอง
for f in sorted(glob.glob(OUT + "/results/external_*_summary.txt")):
    print(open(f, encoding="utf-8").read())

## 8) ลองให้โมเดลที่ดีที่สุดทายรูปใน test

In [ ]:
model = ultralytics.YOLO(OUT + "/models/best_detect.pt")
test_images = sorted(glob.glob("/content/data_5fruits/test/images/*.jpg"))[:4]
model.predict(test_images, conf=0.40, save=True, project="/content/demo", name="pred", exist_ok=True, verbose=False)
for f in sorted(glob.glob("/content/demo/pred/*.jpg")):
    display(Image(f, width=500))

## 9) เอาไปใช้ต่อ
- `KhadSan_detect/models/best_detect.pt` + `class_info.json` → วางใน `web/models/` ของโปรเจกต์
- `KhadSan_detect/results/` → วางใน `results/` ของโปรเจกต์ (กราฟ ตาราง ใส่รายงาน/สไลด์)
- ไม่ต้องโหลด `runs/` (ไฟล์ใหญ่ เก็บไว้ใน Drive พอ)